In [12]:
import onnxruntime as ort
import pandas as pd
from pathlib import Path
import os

# Definisi path berdasarkan struktur workspace
ROOT_DIR = Path(os.getcwd())
YOLOV11_DIR = ROOT_DIR / "models" / "yolov11" / "simplified"
OWLV2_MODEL_PATH = ROOT_DIR / "models" / "owlv2" / "owlv2.onnx"

def get_model_info(model_name, model_path):
    """Mengambil informasi metadata dari model ONNX."""
    if not model_path.exists():
        print(f"⚠️ File tidak ditemukan: {model_path}")
        return None

    try:
        # Load model ONNX
        session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
        
        # Informasi Input
        input_info = []
        for inp in session.get_inputs():
            shape = str(inp.shape)
            input_info.append(f"{inp.name} {shape}")
            
        # Informasi Output
        output_info = []
        for out in session.get_outputs():
            shape = str(out.shape)
            output_info.append(f"{out.name} {shape}")
            
        # Metadata tambahan
        meta = session.get_modelmeta()
        
        return {
            "Nama Model": model_name,
            "Input Layer": "\n".join(input_info),
            "Output Layer": "\n".join(output_info),
            "Penyedia": meta.producer_name,
            "Versi IR": meta.version,
        }
        
    except Exception as e:
        print(f"❌ Error memeriksa model {model_name}: {e}")
        return None

# List untuk menyimpan data
data_models = []

# Periksa model YOLOv11 (n, s, m)
for size in ["n", "s", "m"]:
    file_path = YOLOV11_DIR / f"yolo11{size}.onnx"
    info = get_model_info(f"YOLOv11-{size}", file_path)
    if info:
        data_models.append(info)

# Periksa model OWLv2
info_owl = get_model_info("OWLv2", OWLV2_MODEL_PATH)

# Buat DataFrame untuk tampilan yang rapi
df_models = pd.DataFrame(data_models)

# Tampilkan DataFrame
print("Ringkasan Spesifikasi Model ONNX:")
print(df_models)

Ringkasan Spesifikasi Model ONNX:
  Nama Model              Input Layer          Output Layer Penyedia  \
0  YOLOv11-n  images [1, 3, 416, 416]  output0 [1, 5, 3549]  pytorch   
1  YOLOv11-s  images [1, 3, 416, 416]  output0 [1, 5, 3549]  pytorch   
2  YOLOv11-m  images [1, 3, 416, 416]  output0 [1, 5, 3549]  pytorch   

              Versi IR  
0  9223372036854775807  
1  9223372036854775807  
2  9223372036854775807  


In [13]:
print(f"=== Model: {info_owl['Nama Model']} ===")
for key, value in info_owl.items():
    if key == 'Nama Model': continue
    print(f"\n[{key}]")
    if isinstance(value, str) and '\n' in value:
        # Pecah string multi-line dan beri indentasi
        for line in value.split('\n'):
            print(f"  • {line}")
    else:
        print(f"  {value}")

=== Model: OWLv2 ===

[Input Layer]
  • input_ids ['text_batch_size', 'sequence_length']
  • pixel_values ['image_batch_size', 'num_channels', 'height', 'width']
  • attention_mask ['text_batch_size', 'sequence_length']

[Output Layer]
  • logits ['image_batch_size', 3600, 'floor(text_batch_size/image_batch_size)']
  • pred_boxes ['image_batch_size', 3600, 4]
  • text_embeds ['image_batch_size', 'floor(text_batch_size/image_batch_size)', 512]
  • image_embeds ['image_batch_size', 60, 60, 768]

[Penyedia]
  onnx.quantize

[Versi IR]
  9223372036854775807
